## 과제 2 — 성과품 정합성 검증 미니 엔진

**왜 이걸 하나:** 우리 시스템의 핵심은 서로 다른 성과품을 하나의 스키마로 모아 모순을 잡아내는
것입니다. `consistency_check.xlsx`에는 두 시트(`Equipment_List`, `Line_List`)가 있고, **서로 다른
엔지니어가 만들었다는 설정**이라 같은 속성이 다른 이름·다른 단위로 들어있습니다. 또한 일부러
모순이 심어져 있습니다.

**해야 할 것 (필수):**
1. 두 시트를 **하나의 공통 스키마(미니 온톨로지)** 로 읽어 들이세요.
2. **이름이 다르지만 같은 속성**을 자동으로 매칭하세요. 단위가 다르면 환산해서 맞추세요.
   (룰베이스든 LLM이든 자유. 단 **왜 그 방식인지 트레이드오프**를 적어주세요.)
3. 시트 안 `READ_ME`에 적힌 **자연어 검증 규칙 5개**를 적용해 **위반 리포트**를 만드세요.
   (어떤 장비/배관이 어떤 규칙을 왜 위반했는지.)

**해야 할 것 (가산, 선택):**
4. 두 시트에 걸친(cross-sheet) 모순도 찾아낼 수 있나요? (규칙 5개에 명시되지 않은 것도 환영)
5. 자연어 규칙을 실행 가능한 형태로 바꿀 때, **잘못 변환됐는지 어떻게 잡아낼지**(검증 단계)를
   설명하세요.

---

## 과제2. 성과품 정합성 검증 미니 엔진

1) Equipment_List와 Line_List를 하나의 공통 스키마로 변환

2) 서로 다른 컬럼명/단위를 자동 매칭 및 환산

3) READ_ME의 자연어 규칙 R1~R5를 코드로 검증

4) 위반 사항을 리포트 형태의 DataFrame으로 출력

##### 1. 엑셀 파일 읽기

In [1]:
import pandas as pd
import numpy as np
import re
from difflib import get_close_matches


file_path = "./consistency_check.xlsx"

equipment_raw = pd.read_excel(file_path, sheet_name="Equipment_List")
line_raw = pd.read_excel(file_path, sheet_name="Line_List")
readme = pd.read_excel(file_path, sheet_name="READ_ME", header=8)

display(equipment_raw.head())
display(line_raw.head())
display(readme.head())

,Equipment Tag,Type,Operating Pressure (barg),Design Pressure (barg),Operating Temp (degC),Design Temp (degC),Material,Has Safety Valve
0,T-101,Storage Tank,1,1.05,40,70,CS,N
1,P-101A,Centrifugal Pump,5,6.00,40,90,CS,N
2,P-101B,Centrifugal Pump,5,5.20,40,90,CS,N
3,E-101,Heat Exchanger,8,10.00,120,160,SS304,Y
4,E-102,Heat Exchanger,6,8.00,110,150,SS304,N


,배관 번호,From 장비,To 장비,설계압력 [kPag],운전온도 [K],보온 여부
0,L-001,T-101,P-101A,105,313,N
1,L-002,T-101,P-101B,105,313,N
2,L-010,P-101A,E-101,600,313,Y
3,L-011,P-101C,E-101,600,313,Y
4,L-020,E-101,V-201,800,393,Y


,Unnamed: 0
0,R1. 모든 장비의 설계압력은 운전압력의 1.1배 이상이어야 한다.
1,R2. 배관 목록(From/To)에서 참조하는 모든 장비 태그는 장비 목록에 존재해...
2,R3. 모든 장비의 설계온도는 운전온도보다 30℃ 이상 높아야 한다(설계온도 ≥ 운...
3,R4. 안전밸브가 있는 장비는 설계압력이 반드시 정의(공란이 아님)되어 있어야 한다.
4,R5. 모든 장비 태그는 다음 패턴을 따라야 한다: 영문자 1자 이상 + 하이픈 +...


#### 2. 실제 파일 컬럼명을 반영한 컬럼 매핑 정의

서로 다른 엔지니어가 만든 시트라서 같은 의미의 컬럼이 영어/한글/단위 포함 형태로 다르게 표현되어 있음.   
따라서 공통 스키마 이름을 기준으로 후보 키워드를 정의한다.

- Equipment_List 컬럼
  - Equipment Tag
  - Type
  - Operating Pressure (barg)
  - Design Pressure (barg)
  - Operating Temp (degC)
  - Design Temp (degC)
  - Material
  - Has Safety Valve

- Line_List 컬럼
  - 배관 번호
  - From 장비
  - To 장비
  - 설계압력 [kPag]
  - 운전온도 [K]
  - 보온 여부

In [2]:
equipment_col_map = {
    "Equipment Tag": "equipment_tag",
    "Type": "equipment_type",
    "Operating Pressure (barg)": "operating_pressure_barg",
    "Design Pressure (barg)": "design_pressure_barg",
    "Operating Temp (degC)": "operating_temp_degC",
    "Design Temp (degC)": "design_temp_degC",
    "Material": "material",
    "Has Safety Valve": "has_safety_valve"
}

line_col_map = {
    "배관 번호": "line_no",
    "From 장비": "from_equipment",
    "To 장비": "to_equipment",
    "설계압력 [kPag]": "design_pressure_kPag",
    "운전온도 [K]": "operating_temp_K",
    "보온 여부": "insulation"
}


equipment = equipment_raw.rename(columns=equipment_col_map).copy()
line = line_raw.rename(columns=line_col_map).copy()

display(equipment.head())
display(line.head())

,equipment_tag,equipment_type,operating_pressure_barg,design_pressure_barg,operating_temp_degC,design_temp_degC,material,has_safety_valve
0,T-101,Storage Tank,1,1.05,40,70,CS,N
1,P-101A,Centrifugal Pump,5,6.00,40,90,CS,N
2,P-101B,Centrifugal Pump,5,5.20,40,90,CS,N
3,E-101,Heat Exchanger,8,10.00,120,160,SS304,Y
4,E-102,Heat Exchanger,6,8.00,110,150,SS304,N


,line_no,from_equipment,to_equipment,design_pressure_kPag,operating_temp_K,insulation
0,L-001,T-101,P-101A,105,313,N
1,L-002,T-101,P-101B,105,313,N
2,L-010,P-101A,E-101,600,313,Y
3,L-011,P-101C,E-101,600,313,Y
4,L-020,E-101,V-201,800,393,Y


#### 3. 단위 환산 

압력 단위 통일(barg) / 온도 단위 통일(degC)
- 1 barg = 100 kPag
- K = ℃ + 273.15

In [3]:
line["design_pressure_barg"] = line["design_pressure_kPag"] / 100
line["operating_temp_degC"] = line["operating_temp_K"] - 273.15

display(equipment.head())
display(line.head())

,equipment_tag,equipment_type,operating_pressure_barg,design_pressure_barg,operating_temp_degC,design_temp_degC,material,has_safety_valve
0,T-101,Storage Tank,1,1.05,40,70,CS,N
1,P-101A,Centrifugal Pump,5,6.00,40,90,CS,N
2,P-101B,Centrifugal Pump,5,5.20,40,90,CS,N
3,E-101,Heat Exchanger,8,10.00,120,160,SS304,Y
4,E-102,Heat Exchanger,6,8.00,110,150,SS304,N


,line_no,from_equipment,to_equipment,design_pressure_kPag,operating_temp_K,insulation,design_pressure_barg,operating_temp_degC
0,L-001,T-101,P-101A,105,313,N,1.05,39.85
1,L-002,T-101,P-101B,105,313,N,1.05,39.85
2,L-010,P-101A,E-101,600,313,Y,6.00,39.85
3,L-011,P-101C,E-101,600,313,Y,6.00,39.85
4,L-020,E-101,V-201,800,393,Y,8.00,119.85


#### 4. 자연어 검정 규칙 

- R1. 모든 장비의 설계압력은 운전압력의 1.1배 이상이어야 한다.

- R2. 배관 목록(From/To)에서 참조하는 모든 장비 태그는 장비 목록에 존재해야 한다.

- R3. 모든 장비의 설계온도는 운전온도보다 30℃ 이상 높아야 한다(설계온도 ≥ 운전온도 + 30)

- R4. 안전밸브가 있는 장비는 설계압력이 반드시 정의(공란이 아님)되어 있어야 한다.

- R5. 모든 장비 태그는 다음 패턴을 따라야 한다: 영문자 1자 이상 + 하이픈 + 숫자 정확히 3자리, 그 뒤에 병렬 장비를 구분하는 영문자 1자가 선택적으로 올 수 있다 (예: P-101, E-102, V-201, P-101A).

In [4]:
violations = []


def add_violation(rule_id, item_type, item_id, reason):
    """
    검증 규칙 위반 사항을 하나씩 저장하는 함수.
    """
    violations.append({
        "rule_id": rule_id,
        "item_type": item_type,
        "item_id": item_id,
        "reason": reason
    })

In [5]:
# R1. 모든 장비의 설계압력은 운전압력의 1.1배 이상이어야 한다.

for _, row in equipment.iterrows():
    tag = row["equipment_tag"]
    op_p = row["operating_pressure_barg"]
    design_p = row["design_pressure_barg"]

    if pd.isna(op_p):
        add_violation("R1", "Equipment", tag, "운전압력이 공란이므로 설계압력 기준을 계산할 수 없음")

    elif pd.isna(design_p):
        add_violation("R1", "Equipment", tag, "설계압력이 공란이므로 운전압력의 1.1배 이상인지 검증할 수 없음")

    elif design_p < op_p * 1.1:
        add_violation("R1", "Equipment", tag,
                      f"설계압력 {design_p} barg가 운전압력의 1.1배 기준 {op_p*1.1:.2f} barg보다 작음")

In [6]:
# R2. Line_List의 From/To 장비는 Equipment_List에 존재해야 한다.

equipment_tags = set(equipment["equipment_tag"])

for _, row in line.iterrows():
    line_no = row["line_no"]
    from_eq = row["from_equipment"]
    to_eq = row["to_equipment"]

    if from_eq not in equipment_tags:
        add_violation("R2", "Line", line_no, f"From 장비 '{from_eq}'가 Equipment_List에 존재하지 않음")

    elif to_eq not in equipment_tags:
        add_violation("R2", "Line", line_no, f"To 장비 '{to_eq}'가 Equipment_List에 존재하지 않음")

In [7]:
# R3. 모든 장비의 설계온도는 운전온도보다 30℃ 이상 높아야 한다.

for _, row in equipment.iterrows():
    tag = row["equipment_tag"]
    op_t = row["operating_temp_degC"]
    design_t = row["design_temp_degC"]

    if pd.isna(op_t):
        add_violation("R3", "Equipment", tag, "운전온도가 공란이므로 설계온도 기준을 계산할 수 없음")

    elif pd.isna(design_t):
        add_violation("R3", "Equipment", tag, "설계온도가 공란이므로 운전온도 + 30℃ 이상인지 검증할 수 없음")

    elif design_t < op_t + 30:
        add_violation("R3", "Equipment", tag, f"설계온도 {design_t}℃가 운전온도 + 30℃ 기준 {op_t + 30:.2f}℃보다 낮음")

In [8]:
# R4. 안전밸브가 있는 장비는 설계압력이 반드시 정의되어 있어야 한다.

for _, row in equipment.iterrows():
    tag = row["equipment_tag"]
    has_sv = row["has_safety_valve"]
    design_p = row["design_pressure_barg"]

    if has_sv == "Y" and pd.isna(design_p):
        add_violation("R4", "Equipment", tag, "안전밸브가 있는 장비인데 설계압력이 공란임")

In [9]:
# R5. 장비 태그 패턴 검증 / 영문자 1자 이상 + 하이픈 + 숫자 정확히 3자리 + 선택적 영문자 1자

tag_pattern = r"^[A-Za-z]+-\d{3}[A-Za-z]?$"

for _, row in equipment.iterrows():
    tag = row["equipment_tag"]

    if not re.match(tag_pattern, str(tag)):
        add_violation("R5", "Equipment", tag, "장비 태그 형식이 규칙에 맞지 않음")

#### 5. 최종 위반 리포트 생성 및 저장

In [10]:
violation_report = pd.DataFrame(violations)
violation_report = violation_report.sort_values(by=["rule_id", "item_type", "item_id"]).reset_index(drop=True)

violation_report.to_excel("violation_report.xlsx", index=False)

print("총 위반 건수:", len(violation_report))
display(violation_report)

총 위반 건수: 7


,rule_id,item_type,item_id,reason
0,R1,Equipment,P-101B,설계압력 5.2 barg가 운전압력의 1.1배 기준 5.50 barg보다 작음
1,R1,Equipment,T-101,설계압력 1.05 barg가 운전압력의 1.1배 기준 1.10 barg보다 작음
2,R1,Equipment,V-202,설계압력이 공란이므로 운전압력의 1.1배 이상인지 검증할 수 없음
3,R2,Line,L-011,From 장비 'P-101C'가 Equipment_List에 존재하지 않음
4,R3,Equipment,C-301,설계온도 215℃가 운전온도 + 30℃ 기준 230.00℃보다 낮음
5,R4,Equipment,V-202,안전밸브가 있는 장비인데 설계압력이 공란임
6,R5,Equipment,PUMP102,장비 태그 형식이 규칙에 맞지 않음
